In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from pymatgen.io.cif import CifParser

a_B = 0.529  
pi = np.pi
HBAR2_OVER_2ME_eVA2 = 3.80998  

EH_A = 40.5
EH_P = 2.5

KF_COEFF = 3.0   

HALIDES = ["F", "Cl", "Br", "I"]
HALIDE_SP = {"F": 7, "Cl": 7, "Br": 7, "I": 7}   
NOBLE_METALS = {"Cu": 11, "Ag": 11}              
NOBLE_SIGMA = {"CuCl": 1.44, "CuBr": 1.41, "CuI": 1.35,  
               "AgCl": 1.39, "AgBr": 1.20, "AgI": 1.49, "AgF": 1.4}
NOBLE_NC = {"CuCl": 4, "CuBr": 4, "CuI": 4,             
            "AgCl": 6, "AgBr": 6, "AgI": 4, "AgF": 6}             

def _to_float(x):
    
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        x = x[0] if len(x) else None
    if x is None:
        return None
    s = str(x).strip()
    if s in ("?", ".", ""):
        return None
    s = re.sub(r"\(.*\)$", "", s) 
    try:
        return float(s)
    except Exception:
        return None


def _to_int(x):
    f = _to_float(x)
    return None if f is None else int(round(f))


def _to_str(x):
    
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        x = x[0] if len(x) else None
    if x is None:
        return None
    s = str(x).strip()
    if s in ("?", ".", ""):
        return None
    
    if (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
        s = s[1:-1].strip()
    return s if s else None


def read_best_cif_block(cif_path):
    
    parser = CifParser(cif_path)
    d = parser.as_dict()

    best = None
    for name, block in d.items():
        Z = _to_int(block.get("_cell_formula_units_Z"))
        if Z is None:
            continue

        V = _to_float(block.get("_cell_volume"))
        has_atoms = (
            "_atom_site_fract_x" in block
            and block.get("_atom_site_fract_x") not in (None, "?", ".", ["?"], ["."])
        )

        score = (2 if has_atoms else 0) + (1 if V is not None else 0)

        if best is None or score > best[0]:
            best = (score, name, block)

    if best is None:
        raise ValueError("Could not find CIF data block with numeric _cell_formula_units_Z")
    return best[2]


def read_Z_Vcell_sg_from_cif(cif_path):
    
    block = read_best_cif_block(cif_path)

    Z = _to_int(block.get("_cell_formula_units_Z"))
    if Z is None:
        raise ValueError("CIF has no usable _cell_formula_units_Z")

    V = _to_float(block.get("_cell_volume"))
    if V is None:
        a = _to_float(block.get("_cell_length_a"))
        b = _to_float(block.get("_cell_length_b"))
        c = _to_float(block.get("_cell_length_c"))
        alpha = _to_float(block.get("_cell_angle_alpha"))
        beta = _to_float(block.get("_cell_angle_beta"))
        gamma = _to_float(block.get("_cell_angle_gamma"))

        if None in (a, b, c, alpha, beta, gamma):
            raise ValueError("CIF missing _cell_volume and insufficient lattice parameters")

        ar, br, gr = np.deg2rad([alpha, beta, gamma])
        V = a * b * c * np.sqrt(
            1
            + 2 * np.cos(ar) * np.cos(br) * np.cos(gr)
            - np.cos(ar) ** 2
            - np.cos(br) ** 2
            - np.cos(gr) ** 2
        )

    
    sg_name = (
        _to_str(block.get("_symmetry_space_group_name_H-M"))
        or _to_str(block.get("_symmetry_space_group_name_H-M_alt"))
        or _to_str(block.get("_space_group_name_H-M_alt"))
        or _to_str(block.get("_space_group_name_H-M_ref"))
        or _to_str(block.get("_space_group_name_H-M"))
    )

    sg_number = (
        _to_int(block.get("_symmetry_Int_Tables_number"))
        or _to_int(block.get("_space_group_IT_number"))
        or _to_int(block.get("_space_group_IT_number"))
    )

    return Z, float(V), sg_name, sg_number


def read_cif_reliably(cif_path):
    
    parser = CifParser(cif_path)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        structs = parser.parse_structures(primitive=False)
    for s in structs:
        if s and len(s) > 0:
            return s
    raise ValueError("No valid structure in CIF")


def site_symbol(site):
   
    if hasattr(site, "specie") and site.specie is not None:
        return site.specie.symbol
    if hasattr(site, "species") and site.species is not None:
        sp = max(site.species.items(), key=lambda kv: kv[1])[0]
        return sp.symbol
    return str(site).split()[0]


def identify_metal_halide(structure):
    
    elems = [el.symbol for el in structure.composition.elements]
    halides = [e for e in elems if e in HALIDES]
    non_halides = [e for e in elems if e not in HALIDES]

    junk = {"H", "C", "N", "O"}
    metal = next((e for e in non_halides if e not in junk), None) or (non_halides[0] if non_halides else None)
    halide = halides[0] if halides else None
    return metal, halide


def get_mean_mx_bond_length(structure, metal, halide, r_cut=3.6):
    
    bonds = []
    for site in structure:
        if site_symbol(site) == metal:
            for nbor in structure.get_neighbors(site, r_cut):
                if site_symbol(nbor) == halide:
                    bonds.append(nbor.nn_distance)
    return float(np.mean(bonds)) if bonds else 2.6


def parse_stoichiometry(name):
    
    base = re.sub(r"\s+(1T'?|2H|3R)\s*$", "", name.strip())
    parts = re.findall(r"([A-Z][a-z]*)(\d*)", base)
    m, n_atoms = 1, 2
    if len(parts) == 2:
        m = int(parts[0][1]) if parts[0][1] else 1
        n_atoms = int(parts[1][1]) if parts[1][1] else 1
    return m, n_atoms


def compute_ks_from_structure(Vcell, cell_formula_units_Z, valence_sp_per_fu, kf_coeff=1.5):
    
    n = float(valence_sp_per_fu) * float(cell_formula_units_Z) / float(Vcell)
    kF = (kf_coeff * pi**2 * n) ** (1.0 / 3.0)
    ks = np.sqrt(4.0 * kF / (pi * a_B))
    return ks, n, kF



def homopolar_gap(d):
    
    return EH_A / d ** EH_P


def gap(d, Vcell, Z, m, n_atoms, metal, halide, material):
    
    if metal not in NOBLE_METALS:
        raise KeyError(f"{material}: '{metal}' is not a noble metal.")
    if material not in NOBLE_SIGMA:
        raise KeyError(f"{material}: no parameters — add it to NOBLE_SIGMA and NOBLE_NC.")

    halide_sp = HALIDE_SP[halide]
    metal_sp = 1                                               
    valence_sp_per_fu = metal_sp * m + halide_sp * n_atoms

    r0 = d / 2.0
    ks, n, kF = compute_ks_from_structure(Vcell, Z, valence_sp_per_fu, KF_COEFF)
    Ef = HBAR2_OVER_2ME_eVA2 * kF ** 2

    b = 0.089 * NOBLE_NC[material] ** 2                               
    Zstar = NOBLE_SIGMA[material] * NOBLE_METALS[metal]               
    Egc = homopolar_gap(d)                                           
    Egi = 14.4 * b * np.exp(-ks * r0) * abs(Zstar - halide_sp) / r0  
    Eg = np.sqrt(Egc ** 2 + Egi ** 2)
    fi = Egi ** 2 / Eg ** 2                                          
    return {"r0": r0, "n": n, "kF": kF, "ks": ks, "Ef": Ef,
            "Zstar": Zstar, "b": b, "Egc": Egc, "Egi": Egi, "Eg": Eg, "fi": fi}



excel_in = "Bandgaps-Non Layered TMHs-input.xlsx"
cif_dir = "CIFs of Non Layered TMHs"
excel_out = "Bandgaps-Non Layered TMHs-output.xlsx"

df = pd.read_excel(excel_in)
rows = []

for i, row in df.iterrows():
    material = str(row["Material"]).strip()
    cif_path = os.path.join(cif_dir, f"{material}-SM.cif")
    print(f"\n[{i+1}/{len(df)}] {material}")

    if not os.path.exists(cif_path):
        print("   ⚠ CIF not found")
        continue

    try:
       
        cell_formula_units_Z, Vcell, sg_name, sg_number = read_Z_Vcell_sg_from_cif(cif_path)
        s = read_cif_reliably(cif_path)

        metal, halide = identify_metal_halide(s)
        if metal is None or halide is None:
            raise ValueError("Could not identify metal/halide from structure")

        m, n_atoms = parse_stoichiometry(material)
        d = get_mean_mx_bond_length(s, metal, halide, r_cut=3.6)

      
        res = gap(d, Vcell, cell_formula_units_Z, m, n_atoms, metal, halide, material)
        valence_e_per_fu = NOBLE_METALS[metal] * m + HALIDE_SP[halide] * n_atoms

        rows.append({
            "Material": material,
            "symmetry_space_group_name_H-M": sg_name,
            "symmetry_Int_Tables_number": sg_number,
            "cell_formula_units_Z": cell_formula_units_Z,
            "valence_electrons_per_fu": valence_e_per_fu,
            "d (Å)": d,
            "r0 (Å)": res["r0"],
            "Vcell (Å^3)": Vcell,
            "n (Å^-3)": res["n"],
            "kF (Å^-1)": res["kF"],
            "Ks (Å^-1)": res["ks"],
            "Ef (eV)": res["Ef"],
            "Z* (e)": res["Zstar"],
            "b": res["b"],
            "Egc (eV)": res["Egc"],
            "Egi (eV)": res["Egi"],
            "Eg (eV)": res["Eg"],
            "fi": res["fi"],  
           
        })

        print(
            f"   ✅ Z={cell_formula_units_Z}, Vcell={Vcell:.3f} Å³, SG={sg_name} ({sg_number}), "
            f"Z*={res['Zstar']:.2f}, C={res['Egi']:.2f} eV, Eg={res['Eg']:.2f} eV, f_i={res['fi']:.3f}"
        )

    except Exception as e:
        print(f"   ❌ Error: {e}")

pd.DataFrame(rows).to_excel(excel_out, index=False)
print(f"\n✅ Done. Results saved as '{excel_out}'")


[1/6] AgCl
   ✅ Z=4, Vcell=170.900 Å³, SG=Fm-3m (225), Z*=15.29, C=15.74 eV, Eg=16.05 eV, f_i=0.961

[2/6] AgBr
   ✅ Z=4, Vcell=192.500 Å³, SG=Fm-3m (225), Z*=13.20, C=10.68 eV, Eg=11.05 eV, f_i=0.933

[3/6] AgI
   ✅ Z=4, Vcell=274.000 Å³, SG=F-43m (216), Z*=16.39, C=9.36 eV, Eg=9.85 eV, f_i=0.904

[4/6] CuCl
   ✅ Z=4, Vcell=158.900 Å³, SG=F-43m (216), Z*=15.84, C=13.34 eV, Eg=14.18 eV, f_i=0.885

[5/6] CuBr
   ✅ Z=4, Vcell=182.900 Å³, SG=F-43m (216), Z*=15.51, C=11.57 eV, Eg=12.33 eV, f_i=0.880

[6/6] CuI
   ✅ Z=4, Vcell=222.300 Å³, SG=F-43m (216), Z*=14.85, C=9.20 eV, Eg=9.89 eV, f_i=0.865

✅ Done. Results saved as 'Bandgaps-Non Layered TMHs-output.xlsx'
